# «Разработка интеллектуальной системы озвучивания художественных текстов с алгоритмическим улучшением просодии на базе предобученной TTS-модели»


# 📚 Интеллектуальная система озвучивания аудиокниг с алгоритмическим улучшением просодии

**Автор:** Андреев Вячеслав Михайлович  
**Направление:** AI/ML/TTS  
**Технологии:** Silero TTS v5, PyTorch, pymorphy3, Gradio  
**Гипотеза:** Алгоритмическая расстановка пауз на основе синтаксического анализа улучшает субъективное качество (MOS) озвучки художественного текста без переобучения TTS-модели.

In [ ]:
!pip install torch torchaudio soundfile numpy pymorphy3 gradio pandas tqdm

import torch
import soundfile as sf
import numpy as np
import re
import pymorphy3
import pandas as pd
from typing import List
import gradio as gr

print("✅ Все библиотеки успешно импортированы")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 681.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 22.7 MB/s eta 0:00:00
✅ Все библиотеки успешно импортированы


## 1. Исследование API Silero TTS v5
В ходе отладки API зафиксированы критические параметры:
- **Метод синтеза:** `model.apply_tts(text, sample_rate=16000)`
- **Формат возврата:** `list[torch.Tensor]` с batch-измерением `(1, N)` → требует конвертации в 1D `numpy.ndarray`
- **Sample Rate:** `16000 Гц` (48k ускоряет речь в 3x, 8k замедляет)
- **Лимит входа:** ~140 символов на вызов (превышение вызывает `UserWarning` и артефакты)

In [ ]:
# Загрузка модели
print("🔄 Загрузка Silero TTS v5...")
model, example_text = torch.hub.load(
    repo_or_dir='snakers4/silero-models',
    model='silero_tts',
    language='ru',
    speaker='aidar_v2',
    trust_repo=True
)
print("✅ Модель загружена")

# 🔍 Пояснение: символы "+" в example_text — это фонетические маркеры ударения,
# используемые в тренировочных датасетах. В рабочем пайплайне модель определяет
# ударения автоматически на основе контекста.

# Базовый синтез с правильной конвертацией (исправление багов API)
raw_audio = model.apply_tts(example_text, sample_rate=16000)

# Конвертация: list -> Tensor -> numpy -> 1D
if isinstance(raw_audio, list):
    audio = raw_audio[0]
if isinstance(audio, torch.Tensor):
    audio = audio.detach().cpu().numpy()
if len(audio.shape) == 2:
    audio = audio.squeeze(0)  # Удаление batch dimension

# Нормализация
if np.max(np.abs(audio)) > 1.0:
    audio = audio / np.max(np.abs(audio))

sf.write("base_test.wav", audio.astype(np.float32), 16000)
print(f"✅ Базовый синтез успешен! Длительность: {len(audio)/16000:.2f} сек")

🔄 Загрузка Silero TTS v5...
Downloading: "https://github.com/snakers4/silero-models/zipball/master" to /root/.cache/torch/hub/master.zip


100%|██████████| 132M/132M [00:14<00:00, 9.55MB/s]


✅ Модель загружена
✅ Базовый синтез успешен! Длительность: 4.12 сек


## 2. NLP-препроцессинг: алгоритмическая расстановка пауз
Модуль анализирует пунктуацию и синтаксис, вставляя теги пауз, совместимые с Silero:
- `[spk=0.3]` — запятая
- `[spk=0.6]` — точка с запятой, тире
- `[spk=1.0]` — конец предложения
- `[spk=1.5]` — абзац, многоточие

⚠️ Порядок регулярных выражений критичен: многоточие обрабатывается первым, иначе разобьётся на 3 точки.

In [ ]:
class TextPreprocessor:
    PAUSE_SHORT = "[spk=0.3]"
    PAUSE_MEDIUM = "[spk=0.6]"
    PAUSE_LONG = "[spk=1.0]"
    PAUSE_XLONG = "[spk=1.5]"

    def __init__(self):
        self.morph = pymorphy3.MorphAnalyzer()

    def preprocess(self, text: str) -> str:
        text = self._clean_text(text)
        text = self._add_punctuation_pauses(text)
        return text

    def _clean_text(self, text: str) -> str:
        text = text.replace('...', '…')
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\s+([.,!?;:…])', r'\1', text)
        return text.strip()

    def _add_punctuation_pauses(self, text: str) -> str:
        text = re.sub(r'…\s*', f' {self.PAUSE_XLONG} ', text)
        text = re.sub(r'([.!?])(\s|$)', f'\\1{self.PAUSE_LONG}\\2', text)
        text = re.sub(r',(\s)', f',{self.PAUSE_SHORT}\\1', text)
        text = re.sub(r'([;:])(\s)', f'\\1{self.PAUSE_MEDIUM}\\2', text)
        text = re.sub(r'\s*—\s*', f' {self.PAUSE_MEDIUM} ', text)
        text = re.sub(r'\n\s*\n', f' {self.PAUSE_XLONG} ', text)
        return text

    def split_into_chunks(self, text: str, max_chars: int = 100) -> List[str]:
        chunks, current = [], ""
        parts = re.split(r'(\[spk=[\d.]+\])', text)
        for part in parts:
            if not part.strip(): continue
            if part.startswith('[spk='):
                current += part; continue
            if len(current) + len(part) <= max_chars:
                current += part
            else:
                if current.strip(): chunks.append(current.strip())
                current = part
        if current.strip(): chunks.append(current.strip())
        return chunks

preprocessor = TextPreprocessor()
print("✅ NLP-препроцессор инициализирован")

✅ NLP-препроцессор инициализирован


### Эксперимент: оптимизация `max_chars`
Silero рекомендует ≤140 символов. Но теги добавляют ~27% к длине.

| Итерация | max_chars | Предупреждений | Статус |
|----------|-----------|----------------|--------|
| 1        | 500       | 1              | ❌     |
| 2        | 140       | 1              | ❌     |
| **3**    | **100**   | **0**          | ✅     |

*Формула:* `max_chars = 140 - (4 тега × 10 симв.) ≈ 100`

In [ ]:
class AudiobookPipeline:
    def __init__(self, speaker='aidar_v2', sample_rate=16000):
        self.sample_rate = sample_rate
        self.preprocessor = preprocessor  # Используем глобальный экземпляр
        self.model = model                # Используем глобальную модель

    def synthesize(self, text: str, output_path: str = "output.wav", use_preprocessor: bool = True):
        mode = '🟢 NLP (с паузами)' if use_preprocessor else '🔴 Базовый (без пауз)'
        print(f"\n📝 Текст: {len(text)} симв. | Режим: {mode}")

        if use_preprocessor:
            processed = self.preprocessor.preprocess(text)
            chunks = self.preprocessor.split_into_chunks(processed, max_chars=100)
        else:
            sentences = re.split(r'(?<=[.!?])\s+', text)
            chunks, cur = [], ""
            for s in sentences:
                if len(cur) + len(s) <= 130: cur += s + " "
                else:
                    if cur: chunks.append(cur.strip())
                    cur = s
            if cur: chunks.append(cur.strip())

        audio_chunks = []
        for i, chunk in enumerate(chunks, 1):
            audio = self.model.apply_tts(chunk, sample_rate=self.sample_rate)
            if isinstance(audio, list): audio = audio[0]
            if isinstance(audio, torch.Tensor): audio = audio.detach().cpu().numpy()
            if len(audio.shape) == 2: audio = audio.squeeze(0)
            audio_chunks.append(audio)

        full_audio = np.concatenate(audio_chunks)
        if np.max(np.abs(full_audio)) > 1.0:
            full_audio = full_audio / np.max(np.abs(full_audio))

        sf.write(output_path, full_audio.astype(np.float32), self.sample_rate)
        return output_path, len(full_audio) / self.sample_rate

pipeline = AudiobookPipeline()
print("✅ Пайплайн инициализирован")

✅ Пайплайн инициализирован


## 3. Production-интеграция: Gradio A/B-тест
Интерфейс генерирует **два аудиофайла одновременно** для наглядного сравнения:
🔴 **Baseline** (сырой Silero) vs 🟢 **Enhanced** (с NLP-паузами)

⚠️ *Оптимизация для Colab:* убран `gr.Progress()` (вызывает зависание iframe), добавлен `inline=False` (открываем в новой вкладке).

Далее, при получении ссылки типа: Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9093affc8d708182c2.gradio.live откройте ссылку в окне браузера для демонстрации работы системы.


In [ ]:
def gradio_ab_test(text, speaker="aidar_v2"):
    if not text.strip():
        return None, None, "❌ Введите текст"
    if len(text) > 4000:
        return None, None, "❌ Для A/B-теста макс. 4000 симв."

    try:
        print("\n🔴 Генерация базового варианта...")
        raw_path, dur_raw = pipeline.synthesize(text, "gradio_raw.wav", use_preprocessor=False)

        print("\n🟢 Генерация улучшенного варианта...")
        enh_path, dur_enh = pipeline.synthesize(text, "gradio_enhanced.wav", use_preprocessor=True)

        return raw_path, enh_path, f"🔴 {dur_raw:.1f} сек | 🟢 {dur_enh:.1f} сек"
    except Exception as e:
        return None, None, f"❌ Ошибка: {e}"

demo = gr.Interface(
    fn=gradio_ab_test,
    inputs=[
        gr.Textbox(lines=5, label="📝 Текст для озвучивания", placeholder="Вставьте прозу или стихи..."),
        gr.Dropdown(["aidar_v2", "kseniya_v2", "baya_v2"], label="🎤 Голос", value="aidar_v2")
    ],
    outputs=[
        gr.Audio(label="🔴 Базовый синтез", type="filepath"),
        gr.Audio(label="🟢 Улучшенный синтез", type="filepath"),
        gr.Textbox(label="📋 Статус")
    ],
    title="📚 A/B-тест: Озвучивание аудиокниг",
    description="Сравнение базового TTS и алгоритмического улучшения просодии.",
    examples=[
        ["Буря мглою небо кроет,\nВихри снежные крутя;\nТо, как зверь, она завоет,\nТо заплачет, как дитя..."],
        ["Было раннее утро, солнце только начинало подниматься над горизонтом. Лес ещё спал, окутанный лёгким туманом; птицы молчали. Вдруг — где-то вдалеке — послышался странный звук..."]
    ]
)

# ЗАПУСК: share=True для публичной ссылки, inline=False чтобы не зависал встроенный iframe
demo.launch(share=True, debug=True, inline=False)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a984bec393b63a752b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)

🔴 Генерация базового варианта...

📝 Текст: 95 симв. | Режим: 🔴 Базовый (без пауз)

🟢 Генерация улучшенного варианта...

📝 Текст: 95 симв. | Режим: 🟢 NLP (с паузами)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a984bec393b63a752b.gradio.live


## 4. Соответствие требованиям диплома
| Требование | Как закрыто |
|------------|-------------|
| **Не чрезмерно простой** | Композиция: NLP → TTS → пост-обработка + Gradio UI |
| **Научно обоснованные методы** | Rule-based NLP + экспериментальный подбор параметров |
| **Экспериментальное обоснование** | 4 документа в `experiments/` (API, баги, чанки, архитектура) |
| **Production-интеграция** | Веб-сервис Gradio с A/B-тестированием |
| **Разбираться в коде** | Полная отладка API Silero, конвертация тензоров, чанкинг |

✅ **Проект готов к защите.** Все параметры обоснованы, код воспроизводим, система масштабируема.